In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error




In [2]:
train = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")
test = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
sub = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/sample_submission.csv")

print(train.head())
print(test.head())
print(sub.head())




                     Patient  Weeks   FVC    Percent  Age   Sex SmokingStatus
0  ID00133637202223847701934     -2  3195  92.856312   83  Male  Never smoked
1  ID00133637202223847701934      2  3203  93.088817   83  Male  Never smoked
2  ID00133637202223847701934      4  3097  90.008138   83  Male  Never smoked
3  ID00133637202223847701934      6  3171  92.158800   83  Male  Never smoked
4  ID00133637202223847701934      8  3115  90.531272   83  Male  Never smoked
                     Patient  Weeks   FVC    Percent  Age     Sex  \
0  ID00014637202177757139317      0  3807  90.076661   56    Male   
1  ID00019637202178323708467     13  2100  92.858722   83  Female   
2  ID00047637202184938901501      2  3313  89.929425   68    Male   
3  ID00082637202201836229724     19  2918  99.794802   49  Female   
4  ID00126637202218610655908     18  2375  59.375000   78    Male   

      SmokingStatus  
0         Ex-smoker  
1         Ex-smoker  
2         Ex-smoker  
3  Currently smokes  
4      

In [3]:
train.drop_duplicates(keep=False, inplace=True, subset=["Patient", "Weeks"])

sub["Patient"] = sub["Patient_Week"].apply(lambda x: x.split("_")[0])
sub["Weeks"] = sub["Patient_Week"].apply(lambda x: int(x.split("_")[-1]))
sub = sub[["Patient", "Weeks", "Confidence", "Patient_Week"]]

sub = sub.merge(test.drop("Weeks", axis=1), on="Patient", how="left")

train["WHERE"] = "train"
test["WHERE"] = "val"
sub["WHERE"] = "test"
data = pd.concat([train, test, sub], ignore_index=True)

data["min_week"] = data["Weeks"]
data.loc[data.WHERE == "test", "min_week"] = np.nan
data["min_week"] = data.groupby("Patient")["min_week"].transform("min")

base = data.loc[data.Weeks == data.min_week][["Patient", "FVC"]].copy()
base = base.rename(columns={"FVC": "min_FVC"})
base["nb"] = 1
base["nb"] = base.groupby("Patient")["nb"].transform("cumsum")
base = base[base.nb == 1].drop(columns="nb")

data = data.merge(base, on="Patient", how="left")
data["base_week"] = data["Weeks"] - data["min_week"]

COLS = ["Sex", "SmokingStatus"]
FE = []
for col in COLS:
    for mod in data[col].unique():
        FE.append(mod)
        data[mod] = (data[col] == mod).astype(int)

data["age"] = (data["Age"] - data["Age"].min()) / (
    data["Age"].max() - data["Age"].min()
)
data["BASE"] = (data["min_FVC"] - data["min_FVC"].min()) / (
    data["min_FVC"].max() - data["min_FVC"].min()
)
data["week"] = (data["base_week"] - data["base_week"].min()) / (
    data["base_week"].max() - data["base_week"].min()
)
data["percent"] = (data["Percent"] - data["Percent"].min()) / (
    data["Percent"].max() - data["Percent"].min()
)

data["week_percent"] = data["week"] * data["percent"]
data["age_percent"] = data["age"] * data["percent"]
data["age_week"] = data["age"] * data["week"]

data["Weeks_norm"] = (data["Weeks"] - data["Weeks"].min()) / (
    data["Weeks"].max() - data["Weeks"].min()
)

# New quadratic features to give the model extra signal
data["age_sq"] = data["age"] ** 2
data["week_sq"] = data["week"] ** 2

FE += [
    "age",
    "percent",
    "week",
    "BASE",
    "age_week",
    "week_percent",
    "age_percent",
    "Weeks_norm",
    "age_sq",
    "week_sq",
]

train_df = data[data.WHERE == "train"]
val_df = data[data.WHERE == "val"]
sub_df = data[data.WHERE == "test"]

print("Feature columns:", FE)
print("Shapes -> train:", train_df.shape, "val:", val_df.shape, "test:", sub_df.shape)




Feature columns: ['Male', 'Female', 'Never smoked', 'Ex-smoker', 'Currently smokes', 'age', 'percent', 'week', 'BASE', 'age_week', 'week_percent', 'age_percent', 'Weeks_norm', 'age_sq', 'week_sq']
Shapes -> train: (1380, 28) val: (18, 28) test: (1908, 28)


In [4]:
X_train = train_df[FE].values
y_train = train_df["FVC"].values

X_val = val_df[FE].values
y_val = val_df["FVC"].values

X_test = sub_df[FE].values

NFOLD = 5
kf = KFold(n_splits=NFOLD, shuffle=True, random_state=42)

pred_train = np.zeros((X_train.shape[0],))
pred_test = np.zeros((X_test.shape[0],))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
    model = GradientBoostingRegressor(
        n_estimators=1300,
        learning_rate=0.05,
        max_depth=4,
        random_state=fold,
    )
    model.fit(X_train[tr_idx], y_train[tr_idx])
    pred_train[val_idx] = model.predict(X_train[val_idx])
    pred_test += model.predict(X_test) / NFOLD
    val_score = mean_absolute_error(y_train[val_idx], pred_train[val_idx])
    print(f"Fold {fold} MAE (validation split): {val_score:.4f}")

overall_mae = mean_absolute_error(y_train, pred_train)
# Increase confidence scaling factor to produce a larger sigma (helps Laplace LL)
sigma_opt = max(overall_mae * 3.5, 70)
print("Global MAE (scaled for confidence):", sigma_opt)




Fold 1 MAE (validation split): 73.9300
Fold 2 MAE (validation split): 71.9302
Fold 3 MAE (validation split): 66.5165
Fold 4 MAE (validation split): 76.1297
Fold 5 MAE (validation split): 90.0049
Global MAE (scaled for confidence): 264.9579159931004


In [5]:
sub_df["FVC_pred"] = pred_test
sub_df["Confidence_pred"] = sigma_opt

sub_df["Confidence_pred"] = np.maximum(sub_df["Confidence_pred"], 70)

submission = sub_df[["Patient_Week", "FVC_pred", "Confidence_pred"]].copy()
submission = submission.rename(
    columns={"FVC_pred": "FVC", "Confidence_pred": "Confidence"}
)

submission = submission.merge(
    sub[["Patient_Week", "FVC", "Confidence"]],
    on="Patient_Week",
    how="left",
    suffixes=("", "_orig"),
)
submission["FVC"] = submission["FVC"].fillna(submission["FVC_orig"])
submission["Confidence"] = submission["Confidence"].fillna(
    submission["Confidence_orig"]
)
submission = submission[["Patient_Week", "FVC", "Confidence"]]

print("Submission preview:")
print(submission.head())




Submission preview:
                   Patient_Week          FVC  Confidence
0  ID00126637202218610655908_-3  2251.546659  264.957916
1  ID00126637202218610655908_-2  2246.862594  264.957916
2  ID00126637202218610655908_-1  2245.071344  264.957916
3   ID00126637202218610655908_0  2233.246432  264.957916
4   ID00126637202218610655908_1  2216.715011  264.957916


/tmp/ipykernel_55/743424714.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df["FVC_pred"] = pred_test
/tmp/ipykernel_55/743424714.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df["Confidence_pred"] = sigma_opt
/tmp/ipykernel_55/743424714.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexi

In [6]:
submission.to_csv("submission.csv", index=False)
print("submission.csv written, rows:", submission.shape[0])

submission.csv written, rows: 1908
